# Using LLMs on ARC Systems: Hands-On Lab

In this notebook we will:
1. Connect to ARC's centralized LLM API and make chat completion requests
2. Explore system prompts, web search, and multi-turn conversations
3. Connect to a dedicated OOD LLM session
4. Use pydantic-ai to get structured output from LLMs (and make them argue about hot dogs)

**Before you begin:** Make sure you have your API key from [llm.arc.vt.edu](https://llm.arc.vt.edu) ready.

---
## Part 1: The ARC LLM API

ARC hosts several large language models and provides an **OpenAI-compatible API** at `llm-api.arc.vt.edu`. We can use the standard `openai` Python library and just point it at ARC's servers instead of OpenAI's.

### Step 1: Install the OpenAI library

In [ ]:
!pip install -q openai

### Step 2: Enter your API key

Paste your API key (starts with `sk-`) in the cell below.

> **Note:** For this workshop, we're putting the key directly in the notebook for simplicity. If you plan to share your notebook or commit it to git, pull the key from an environment variable instead:
> ```python
> import os
> API_KEY = os.environ["LLM_API_KEY"]
> ```

In [ ]:
API_KEY = "sk-YOUR-API-KEY"  # <-- Paste your key here

# If you get a 401 error later, come back and double-check this value.

### Step 3: Create the API client

We create an OpenAI client but point it at ARC's API instead of OpenAI's. Two parameters are all we need:
- `api_key`: Your key from llm.arc.vt.edu
- `base_url`: ARC's API endpoint instead of OpenAI's

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://llm-api.arc.vt.edu/api/v1"
)

print("Client created successfully!")

### Step 4: Your first chat completion

A chat completion sends a list of **messages** to the model and gets a response back. Each message has:
- A **role**: `system` (sets behavior), `user` (your input), or `assistant` (model's response)
- **Content**: the text of the message

In [ ]:
response = client.chat.completions.create(
    model="gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is Virginia Tech known for?"}
    ]
)

print(response.choices[0].message.content)

### Step 5: Exploring the response object

The response contains more than just text. Token counts tell you how much input the model received and how much output it generated. Roughly 1 token per 3/4 of a word.

In [ ]:
print("Model used:       ", response.model)
print("Prompt tokens:    ", response.usage.prompt_tokens)
print("Completion tokens:", response.usage.completion_tokens)
print("Total tokens:     ", response.usage.total_tokens)

### Step 6: Changing behavior with the system prompt

The **system message** shapes how the model responds. Changing it changes the answer to the same question.

In [ ]:
question = "Explain what a neural network is."

response_general = client.chat.completions.create(
    model="gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a friendly teacher. Explain concepts simply using everyday analogies. Keep answers to 2-3 sentences."},
        {"role": "user", "content": question}
    ]
)

response_technical = client.chat.completions.create(
    model="gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a computer science professor. Use precise technical language. Keep answers to 2-3 sentences."},
        {"role": "user", "content": question}
    ]
)

print("=== General Audience ===")
print(response_general.choices[0].message.content)
print()
print("=== Technical Audience ===")
print(response_technical.choices[0].message.content)

### Exercise: Try your own prompt

Modify the cell below to ask the model something related to your research or coursework. Try changing the system prompt to control the response style.

In [ ]:
response = client.chat.completions.create(
    model="gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},  # <-- Try changing this
        {"role": "user", "content": "Your question here"}               # <-- Put your question here
    ]
)

print(response.choices[0].message.content)

### Step 7: Web search

LLMs only know what was in their training data. They can't answer questions about recent events. ARC's API supports **web search**, which lets the model look up current information before responding.

First, ask without web search. Then the same question with it enabled.

In [ ]:
search_question = "What workshops is ARC at Virginia Tech offering this semester?"

# Without web search
response_no_search = client.chat.completions.create(
    model="gpt-oss-120b",
    messages=[{"role": "user", "content": search_question}]
)
print("=== Without web search ===")
print(response_no_search.choices[0].message.content)

# With web search
response_with_search = client.chat.completions.create(
    model="gpt-oss-120b",
    messages=[{"role": "user", "content": search_question}],
    extra_body={"tool_ids": ["server:websearch"]}
)
print("\n=== With web search ===")
print(response_with_search.choices[0].message.content)

### Step 8: Multi-turn conversations

LLMs don't have memory between requests. To have a conversation, you include the full message history each time. The model uses all previous messages as context.

In [ ]:
conversation = [
    {"role": "system", "content": "You are a helpful research assistant. Be concise."},
    {"role": "user", "content": "What is the Transformer architecture in machine learning?"}
]

# First turn
response = client.chat.completions.create(model="gpt-oss-120b", messages=conversation)
assistant_reply = response.choices[0].message.content
print("Assistant:", assistant_reply)

# Add the reply to history, then ask a follow-up
conversation.append({"role": "assistant", "content": assistant_reply})
conversation.append({"role": "user", "content": "How does the attention mechanism work in that architecture?"})

# Second turn
response = client.chat.completions.create(model="gpt-oss-120b", messages=conversation)
print("\nAssistant:", response.choices[0].message.content)

# We said "that architecture" without specifying which one.
# The model understood because each request sends the full conversation.

---
## Part 2: Dedicated OOD LLM

The centralized API has rate limits (60 requests/min) and a limited model selection. If you need unrestricted access or a specific model, you can launch a **dedicated LLM** on your own GPU through Open OnDemand.

With a dedicated LLM you get:
- Exclusive access to the GPU, no rate limits
- 40+ models (anything on Hugging Face that ARC has pre-downloaded)
- The same OpenAI-compatible API, so your code stays the same
- Cost: uses service units (SUs) from your allocation

### Before continuing:

1. Open [ood.arc.vt.edu](https://ood.arc.vt.edu) in a **new browser tab**
2. Go to **Interactive Apps** > **LLMs**
3. Select a model, partition (`a30_normal_q` or `l40s_normal_q`), and your account
4. Click **Launch** and wait for the session to start
5. When it's running, note the **connection URL** and **API key** from the session card

> **Note:** OOD sessions can take a few minutes to start. If yours isn't ready yet, follow along with the presenter's session and come back to this section after the workshop.

### Network note

The OOD LLM runs on a **Falcon** GPU node, but this Jupyter session is on **TinkerCliffs**. The two clusters can't talk to each other directly, so we'll set up an **SSH tunnel** to bridge them. The next few cells handle this automatically.

### Connect to your OOD LLM

Your OOD connection URL looks something like `http://fal034:50413/v1`. We need the **hostname** (e.g. `fal034`) and **port** (e.g. `50413`) to set up the SSH tunnel.

In [ ]:
import subprocess, os, time, socket
from urllib.parse import urlparse

OOD_URL = input("Enter your OOD LLM connection URL (e.g. http://fal034:50413/v1): ")
OOD_API_KEY = input("Enter your OOD LLM API key: ")

# Parse the hostname and port from the URL
parsed = urlparse(OOD_URL)
remote_host = parsed.hostname  # e.g. "fal034"
remote_port = parsed.port      # e.g. 50413

# Find a free local port for the tunnel
with socket.socket() as s:
    s.bind(("", 0))
    local_port = s.getsockname()[1]

# Build the SSH tunnel: local_port -> falcon1 -> remote_host:remote_port
tunnel_cmd = [
    "ssh", "-N", "-f",
    "-o", "StrictHostKeyChecking=no",
    "-o", "ExitOnForwardFailure=yes",
    "-L", f"{local_port}:{remote_host}:{remote_port}",
    f"{os.environ['USER']}@falcon1.arc.vt.edu",
]

print(f"Setting up SSH tunnel: localhost:{local_port} -> {remote_host}:{remote_port} via falcon1...")
result = subprocess.run(tunnel_cmd, capture_output=True, text=True)

if result.returncode != 0:
    print(f"ERROR: SSH tunnel failed!\n{result.stderr}")
else:
    time.sleep(1)  # Give tunnel a moment to establish
    # Rewrite the URL to go through the tunnel
    OOD_URL = f"http://localhost:{local_port}{parsed.path}"
    print(f"Tunnel established! Using: {OOD_URL}")

ood_client = OpenAI(api_key=OOD_API_KEY, base_url=OOD_URL)
print("OOD client created!")

### List models and chat

In [ ]:
models = ood_client.models.list()
print("Available models:")
for m in models:
    print(f"  - {m.id}")

In [ ]:
MODEL_NAME = "MODEL_NAME"  # <-- Replace with a model ID from the list above

response = ood_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are three tips for writing a good research abstract?"}
    ]
)

print(response.choices[0].message.content)

### Exercise: Compare the two services

Same prompt to both, timed.

In [ ]:
import time

prompt = "Summarize the key ideas behind reinforcement learning in 3 sentences."

start = time.time()
r1 = client.chat.completions.create(model="gpt-oss-120b", messages=[{"role": "user", "content": prompt}])
t1 = time.time() - start

start = time.time()
r2 = ood_client.chat.completions.create(model=MODEL_NAME, messages=[{"role": "user", "content": prompt}])
t2 = time.time() - start

print(f"=== Centralized API (gpt-oss-120b), {t1:.1f}s ===")
print(r1.choices[0].message.content)
print(f"\n=== OOD Dedicated LLM ({MODEL_NAME}), {t2:.1f}s ===")
print(r2.choices[0].message.content)

---
## Part 3: Structured Output with pydantic-ai

So far we've been getting plain text back from the LLM. That's fine for chat, but if you want to use the output in code (store it in a database, feed it to another function, build a pipeline), you need structured data, not a string you have to parse.

**Pydantic** is a popular Python library for defining data schemas as classes. You declare a class with typed fields, and Pydantic validates that any data you put in matches the types. For example:

```python
class Weather(BaseModel):
    city: str
    temp_f: float
    conditions: str
```

**pydantic-ai** connects this to an LLM. You give it a Pydantic class as the `output_type`, and the LLM is forced to return a valid instance of that class. Instead of parsing free-form text, you get back a Python object where `result.city` is a `str`, `result.temp_f` is a `float`, etc.

To show this off, we're going to make two LLMs argue a ridiculous topic and have a third one judge. The audience picks the topic.

In [ ]:
!pip install -q pydantic-ai nest_asyncio

import nest_asyncio
nest_asyncio.apply()

### Define the structured output

We define exactly what fields we want back. pydantic-ai makes the LLM return a valid Python object matching this schema.

In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


class DebateArgument(BaseModel):
    argument: str = Field(description="Your main argument in 2-3 sentences")
    evidence: list[str] = Field(description="2-3 pieces of supporting evidence")
    snark_level: int = Field(ge=1, le=10, description="How snarky, 1-10")
    confidence: float = Field(ge=0, le=1, description="How confident, 0-1")

class Verdict(BaseModel):
    winner: str = Field(description="FOR or AGAINST")
    reasoning: str = Field(description="Why this side won, 2-3 sentences")
    score_for: int = Field(ge=1, le=10)
    score_against: int = Field(ge=1, le=10)
    best_moment: str = Field(description="Best line from the debate")


def arc_model(name: str) -> OpenAIChatModel:
    return OpenAIChatModel(
        name,
        provider=OpenAIProvider(
            base_url="https://llm-api.arc.vt.edu/api/v1",
            api_key=API_KEY,
        ),
    )

print("Ready!")

### Run the debate

Change the topic to whatever you want. Two models argue, a third judges. All responses come back as structured Python objects.

In [ ]:
TOPIC = "Is a hot dog a sandwich?"  # <-- Change this to whatever you want!

debater_for = Agent(
    arc_model("gpt-oss-120b"),
    output_type=DebateArgument,
    instructions=f"You are debating: {TOPIC}. Argue FOR. Be persuasive, witty, and opinionated. Keep it to 2-3 sentences.",
    retries=5,
)
debater_against = Agent(
    arc_model("gpt-oss-120b"),
    output_type=DebateArgument,
    instructions=f"You are debating: {TOPIC}. Argue AGAINST. Be persuasive, witty, and opinionated. Keep it to 2-3 sentences.",
    retries=5,
)
judge = Agent(
    arc_model("gpt-oss-120b"),
    output_type=Verdict,
    instructions="You are a debate judge. Evaluate on logic, evidence, style, and humor. Be fair but entertaining.",
    retries=5,
)

print(f"Topic: {TOPIC}\n")

# Round 1
print("=" * 60)
print("ROUND 1")
print("=" * 60)

arg_for_1 = debater_for.run_sync(f"Make your opening argument: {TOPIC}").output
print(f"\nFOR:")
print(f"  {arg_for_1.argument}")
print(f"  Evidence: {arg_for_1.evidence}")
print(f"  Snark: {arg_for_1.snark_level}/10 | Confidence: {arg_for_1.confidence:.0%}")

arg_against_1 = debater_against.run_sync(f"Make your opening argument: {TOPIC}").output
print(f"\nAGAINST:")
print(f"  {arg_against_1.argument}")
print(f"  Evidence: {arg_against_1.evidence}")
print(f"  Snark: {arg_against_1.snark_level}/10 | Confidence: {arg_against_1.confidence:.0%}")

# Round 2
print("\n" + "=" * 60)
print("ROUND 2")
print("=" * 60)

arg_for_2 = debater_for.run_sync(
    f"Your opponent argued: \"{arg_against_1.argument}\" with evidence: {arg_against_1.evidence}. Counter them."
).output
print(f"\nFOR:")
print(f"  {arg_for_2.argument}")
print(f"  Evidence: {arg_for_2.evidence}")
print(f"  Snark: {arg_for_2.snark_level}/10 | Confidence: {arg_for_2.confidence:.0%}")

arg_against_2 = debater_against.run_sync(
    f"Your opponent argued: \"{arg_for_2.argument}\" with evidence: {arg_for_2.evidence}. Counter them."
).output
print(f"\nAGAINST:")
print(f"  {arg_against_2.argument}")
print(f"  Evidence: {arg_against_2.evidence}")
print(f"  Snark: {arg_against_2.snark_level}/10 | Confidence: {arg_against_2.confidence:.0%}")

### The verdict

Now the judge evaluates the debate and returns a structured scorecard.

In [ ]:
transcript = (
    f"Round 1:\n"
    f"FOR: {arg_for_1.argument} (evidence: {arg_for_1.evidence})\n"
    f"AGAINST: {arg_against_1.argument} (evidence: {arg_against_1.evidence})\n\n"
    f"Round 2:\n"
    f"FOR: {arg_for_2.argument} (evidence: {arg_for_2.evidence})\n"
    f"AGAINST: {arg_against_2.argument} (evidence: {arg_against_2.evidence})"
)

v = judge.run_sync(
    f"Judge this debate about '{TOPIC}'.\n\n{transcript}"
).output

print("=" * 60)
print("VERDICT (gpt-oss-120b)")
print("=" * 60)
print(f"  Winner:      {v.winner}")
print(f"  Score:       FOR {v.score_for}/10 vs AGAINST {v.score_against}/10")
print(f"  Reasoning:   {v.reasoning}")
print(f"  Best moment: {v.best_moment}")
print("=" * 60)

Every response came back as a typed Python object. `arg_for_1.snark_level` is an `int`, `v.score_for` is an `int`, `arg_against_2.evidence` is a `list[str]`. No parsing, no regex, no hoping the LLM formatted things correctly. You define the schema, the LLM fills it in, pydantic validates it.

This is useful any time you need LLM output in a structured format: extracting metadata from papers, generating config files, building data pipelines, or making two robots argue about hot dogs.

---
## Cleanup

The cell below kills the SSH tunnel we opened for the OOD LLM connection.

In [ ]:
# Kill the SSH tunnel
!pkill -f "ssh.*-L.*falcon1.arc.vt.edu"
print("SSH tunnel closed.")

When you're done:
1. Delete your OOD LLM session: go to [ood.arc.vt.edu](https://ood.arc.vt.edu), find the session, and click **Delete**
2. Delete your Jupyter session the same way
3. Simply closing the browser tab does **NOT** stop these jobs

### Resources

- ARC AI docs: [docs.arc.vt.edu/zz_ai.html](https://docs.arc.vt.edu/zz_ai.html)
- ARC Office Hours: [arc.vt.edu/about/office-hours.html](https://arc.vt.edu/about/office-hours.html)
- ARC Support: [arc.vt.edu/help](https://arc.vt.edu/help)